# Ejercicio 1 - Maximización Cúbica con Algoritmos Genéticos

En este notebook implementa el Ejercicio 1 de la guía de laboratorio: maximizar la función

$$f(x) = x^3 - 4x^2 + 5x$$

El objetivo es a raiz de la clase vista, implementar las funciones, variables y buen manejo de documentación para el desarrollo de esta actividad. La cual esta contemplada por partes, trabajada dentro del repositorio de Github en colaboracion por grupo de trabajo

## Alcance de esta primera entrega

- Definir un espacio de búsqueda acotado donde exista un máximo local claro
- Decodificar cromosomas binarios a valores reales de $x$
- Evaluar la aptitud directamente sobre la función cubica
- Visualizar la evolucion del algoritmo y el dataset de helados como contexto visual
- Dejar la base lista para extender y conectar el Ejercicio 3 sobre la tasa de mutacion

**Dataset:** "https://www.kaggle.com/datasets/mirajdeepbhandari/polynomial-regression"
El dataset consiste en un reporte de temperaturas vs ventas, siendo una relacion no lineal con curvatura clara, perfecto para aplicar la busqueda de coeficientes para AG y aplicacion de la funcion matematica

Ejercicio1 Elaborado por Nicolas Ballesteros

## 1. Configuración e imports

Los pasos iniciales en este ejercicio consisten en crear un entorno virtual venv junto a Jupyter, aplicando python como lenguaje base para compilar el codigo a continuacion. En este ejercicio se usa la version 3.11.15 de Python. Para inicializar, cargamos las librerias principales a utilizar `pandas`, `numpy`, `matplotlib` e `ipywidgets`. Para el dataset, se importa de manera diferente. En entornos locales se invoca dentro de la funcion en el codigo mediante su ruta y nombre del archivo. En caso del entorno de colab en la nube requiere subirse a Google Drive e importar las librerias de google, a la vez que se debe definir la ruta exacta del archivo en cuestion para ser utilizado en el notebook. Dicho esto, a continuacion las librerias a utilizar:

In [1]:
import copy
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    import ipywidgets as widgets
    from IPython.display import display
except ImportError:
    widgets = None
    display = None

random.seed(42)
np.random.seed(42)

plt.style.use("seaborn-v0_8-whitegrid")

print("Imports cargados correctamente.")

Imports cargados correctamente.


## 2. Problema a optimizar

Para este ejercicio se usa un rango acotado donde el maximo local de la funcion cubica quede claro y sea facil de interpretar visualmente. En este notebook se trabajara por defecto con:

- `x_min = 0.0`
- `x_max = 1.55`

Con ese intervalo, el punto crítico en `x = 1` se comporta como el mejor valor de la region explorada

Los valores se pueden ajustar y modificar para obtener variaciones

In [ ]:
x_min = 0.0
x_max = 1.55
n_bits = 16
population_size = 40
num_generations = 60
pc = 0.85
pm = 0.03
selection_method = "roulette"
elitism = True


def fitness_cubica(x: float) -> float:
    """calcula la aptitud de la funcion objetivo del ejercicio 1"""
    return x**3 - 4 * x**2 + 5 * x


def decode_binary_to_real(chromosome: list[int], lower_bound: float = x_min, upper_bound: float = x_max) -> float:
    """decodifica un cromosoma binario a un valor real dentro del intervalo definido"""
    integer_value = int("".join(map(str, chromosome)), 2)
    max_integer = (2**len(chromosome)) - 1
    return lower_bound + (integer_value / max_integer) * (upper_bound - lower_bound)


def load_context_dataset() -> pd.DataFrame:
    """Carga el dataset de helados desde la carpeta local del proyecto | Si no existe, devuelve un DataFrame vacío con las columnas esperadas"""
    dataset_path = Path("Dataset") / "Ice_cream selling data.csv"
    if dataset_path.exists():
        return pd.read_csv(dataset_path)
    return pd.DataFrame(columns=["Temperature (°C)", "Ice Cream Sales (units)"])


dataset_helados = load_context_dataset()
dataset_helados.head()

,Temperature (°C),Ice Cream Sales (units)
0,-4.662263,41.842986
1,-4.316559,34.661120
2,-4.213985,39.383001
3,-3.949661,37.539845
4,-3.578554,32.284531
